# C03 Label Smoothing + GroupDRO

Challenger experiment: hybridize the two strongest directions seen so far: label smoothing for calibration and GroupDRO for worst-group pressure. `city swap` stays a separate evaluation step. The grid is intentionally tight around the values that already looked strongest on earlier runs.

In [1]:
import json, random, numpy as np, pandas as pd, torch, joblib
import torch.nn.functional as F
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
CWD = Path.cwd()
NOTEBOOKS_DIR = CWD.parent if CWD.name == "challengers" else CWD
PROJECT_ROOT = NOTEBOOKS_DIR.parent
DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models" / "challengers"
FIGURES_DIR = PROJECT_ROOT / "figures" / "challengers"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"CWD: {CWD}")
print(f"NOTEBOOKS_DIR: {NOTEBOOKS_DIR}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATA_DIR exists: {(DATA_DIR / "train.csv").exists()}")
print(f"MODELS_DIR: {MODELS_DIR}")
print(f"FIGURES_DIR: {FIGURES_DIR}")


CWD: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks/challengers
NOTEBOOKS_DIR: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/notebooks
PROJECT_ROOT: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository
DATA_DIR exists: True
MODELS_DIR: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/models/challengers
FIGURES_DIR: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/figures/challengers


In [3]:
df_train = pd.read_csv(DATA_DIR / "train.csv")
df_val = pd.read_csv(DATA_DIR / "val.csv")
df_test = pd.read_csv(DATA_DIR / "test.csv")

mapping = pd.read_csv(DATA_DIR / "label_to_supercategory_v1.csv")
label_to_supercat = dict(zip(mapping["label"], mapping["supercategory"]))
for df in [df_train, df_val, df_test]:
    df["supercategory"] = df["label"].map(label_to_supercat)

le = LabelEncoder()
df_train["y"] = le.fit_transform(df_train["supercategory"])
df_val["y"] = le.transform(df_val["supercategory"])
df_test["y"] = le.transform(df_test["supercategory"])
num_labels = len(le.classes_)

city_counts = df_train["city_group"].value_counts()
raw_w = 1.0 / np.sqrt(city_counts)
city_weight_map = (raw_w / raw_w.mean()).to_dict()
df_train["sample_weight"] = df_train["city_group"].map(city_weight_map).astype(float)

city_to_id = {c: i for i, c in enumerate(df_train["city_group"].unique())}
num_groups = len(city_to_id)
df_train["city_id"] = df_train["city_group"].map(city_to_id).astype(int)
df_val["city_id"] = df_val["city_group"].map(city_to_id).fillna(-1).astype(int)
df_test["city_id"] = df_test["city_group"].map(city_to_id).fillna(-1).astype(int)

print(f"Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}")
print(f"Labels: {num_labels}, City groups: {num_groups}")


Train: 16530, Val: 5510, Test: 5510
Labels: 9, City groups: 41


In [4]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")


def tokenize(batch):
    return tokenizer(batch["resume_text"], padding="max_length", truncation=True, max_length=128)


train_ds = Dataset.from_pandas(df_train[["resume_text", "y", "city_id", "sample_weight"]]).map(tokenize, batched=True)
val_ds = Dataset.from_pandas(df_val[["resume_text", "y", "city_id"]]).map(tokenize, batched=True)
test_ds = Dataset.from_pandas(df_test[["resume_text", "y", "city_id"]]).map(tokenize, batched=True)

train_ds = train_ds.rename_column("y", "labels")
val_ds = val_ds.rename_column("y", "labels")
test_ds = test_ds.rename_column("y", "labels")

train_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels", "city_id", "sample_weight"])
val_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels", "city_id"])
test_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels", "city_id"])


Map: 100%|██████████| 5510/5510 [00:05<00:00, 1088.80 examples/s]


In [5]:
class LabelSmoothingGroupDROTrainer(Trainer):
    def __init__(self, *args, num_groups, smoothing=0.1, num_classes=9, eta=0.05, **kwargs):
        super().__init__(*args, **kwargs)
        self.num_groups = num_groups
        self.smoothing = smoothing
        self.num_classes = num_classes
        self.eta = eta
        self.q = torch.ones(num_groups) / num_groups
        print(f"Label Smoothing + GroupDRO: eps={smoothing}, eta={eta}, groups={num_groups}")

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        city_id = inputs.pop("city_id")
        sample_weight = inputs.pop("sample_weight", None)
        labels = inputs["labels"]
        outputs = model(**inputs)
        logits = outputs.logits

        log_probs = F.log_softmax(logits, dim=-1)
        smooth_targets = torch.full_like(log_probs, self.smoothing / (self.num_classes - 1))
        smooth_targets.scatter_(1, labels.unsqueeze(1), 1.0 - self.smoothing)
        per_sample_loss = -(smooth_targets * log_probs).sum(dim=-1)
        if sample_weight is not None:
            per_sample_loss = per_sample_loss * sample_weight.to(per_sample_loss.dtype)

        device = per_sample_loss.device
        group_loss = torch.zeros(self.num_groups, device=device)
        group_count = torch.zeros(self.num_groups, device=device)
        for g in range(self.num_groups):
            mask = city_id == g
            if mask.any():
                group_loss[g] = per_sample_loss[mask].mean()
                group_count[g] = mask.sum().float()

        with torch.no_grad():
            q = self.q.to(device) * torch.exp(self.eta * group_loss * (group_count > 0).float())
            self.q = (q / q.sum()).cpu()

        loss = (self.q.to(device) * group_loss).sum()
        return (loss, outputs) if return_outputs else loss


def compute_metrics(prediction_output):
    preds = np.argmax(prediction_output.predictions, axis=1)
    return {
        "accuracy": accuracy_score(prediction_output.label_ids, preds),
        "macro_f1": f1_score(prediction_output.label_ids, preds, average="macro"),
    }


def ovr_rates(df, group_col, num_classes):
    groups = sorted(df[group_col].dropna().unique())
    tpr = np.zeros((len(groups), num_classes))
    support = np.zeros((len(groups), num_classes))
    for gi, group_name in enumerate(groups):
        dg = df[df[group_col] == group_name]
        yt, yp = dg["y_true"].values, dg["y_pred"].values
        for c in range(num_classes):
            positive_mask = yt == c
            tp = np.sum((yp == c) & positive_mask)
            fn = np.sum((yp != c) & positive_mask)
            support[gi, c] = positive_mask.sum()
            tpr[gi, c] = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    return tpr, support


def robust_gaps(tpr, support, min_support=30):
    gaps = []
    for c in range(tpr.shape[1]):
        col = tpr[support[:, c] >= min_support, c]
        col = col[~np.isnan(col)]
        gaps.append(col.max() - col.min() if len(col) >= 2 else np.nan)
    gaps = np.array(gaps)
    valid = gaps[~np.isnan(gaps)]
    return (valid.max() if len(valid) else np.nan, valid.mean() if len(valid) else np.nan)


In [6]:
RUNS = [
    {"tag": "eps005_eta005_3ep", "smoothing": 0.05, "eta": 0.05, "epochs": 3},
    {"tag": "eps01_eta005_3ep", "smoothing": 0.10, "eta": 0.05, "epochs": 3},
    {"tag": "eps01_eta01_3ep", "smoothing": 0.10, "eta": 0.10, "epochs": 3},
]

summary_rows = []

for run in RUNS:
    model_name = f"ls_gdro_{run['tag']}"
    save_dir = MODELS_DIR / model_name
    checkpoint_dir = save_dir / "checkpoints"

    print("\n" + "=" * 80)
    print(f"Training {model_name}: eps={run['smoothing']}, eta={run['eta']}, epochs={run['epochs']}")

    model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=num_labels)
    args = TrainingArguments(
        output_dir=str(checkpoint_dir),
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=run["epochs"],
        weight_decay=0.01,
        warmup_ratio=0.1,
        logging_steps=100,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        remove_unused_columns=False,
        report_to="none",
        seed=SEED,
    )

    trainer = LabelSmoothingGroupDROTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
        num_groups=num_groups,
        smoothing=run["smoothing"],
        num_classes=num_labels,
        eta=run["eta"],
    )

    trainer.train()
    pred_output = trainer.predict(test_ds)
    y_true = pred_output.label_ids
    y_pred = np.argmax(pred_output.predictions, axis=1)

    acc = float(accuracy_score(y_true, y_pred))
    macro_f1 = float(f1_score(y_true, y_pred, average="macro"))

    df_eval = df_test.copy()
    df_eval["y_true"] = y_true
    df_eval["y_pred"] = y_pred
    tpr, support = ovr_rates(df_eval, "city_group", num_labels)
    worst_gap, macro_gap = robust_gaps(tpr, support, min_support=30)

    save_dir.mkdir(parents=True, exist_ok=True)
    trainer.model.save_pretrained(save_dir, safe_serialization=True)
    tokenizer.save_pretrained(save_dir)
    joblib.dump(le, save_dir / "label_encoder.joblib")

    config = {
        "method": "Label Smoothing + GroupDRO + sqrt_rw",
        "smoothing": run["smoothing"],
        "eta": run["eta"],
        "epochs": run["epochs"],
        "learning_rate": 2e-5,
        "weight_decay": 0.01,
        "warmup_ratio": 0.1,
        "accuracy": acc,
        "macro_f1": macro_f1,
        "tpr_gap_worst_robust": float(worst_gap),
        "tpr_gap_macro_robust": float(macro_gap),
    }
    with open(save_dir / "training_config.json", "w") as f:
        json.dump(config, f, indent=2)

    summary_rows.append({
        "model_name": model_name,
        "smoothing": run["smoothing"],
        "eta": run["eta"],
        "epochs": run["epochs"],
        "accuracy": acc,
        "macro_f1": macro_f1,
        "worst_gap": float(worst_gap),
        "macro_gap": float(macro_gap),
        "model_dir": str(save_dir.relative_to(PROJECT_ROOT)),
    })

summary_df = pd.DataFrame(summary_rows).sort_values(["worst_gap", "macro_f1"], ascending=[True, False]).reset_index(drop=True)
summary_path = FIGURES_DIR / "c03_label_smoothing_groupdro_summary.csv"
summary_df.to_csv(summary_path, index=False)
summary_df



Training ls_gdro_eps005_eta005_3ep: eps=0.05, eta=0.05, epochs=3


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1436.28it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those p

Label Smoothing + GroupDRO: eps=0.05, eta=0.05, groups=41


/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.232693,1.234572,0.539383,0.563630
2,0.218901,1.147945,0.587477,0.608348
3,0.220463,1.146222,0.594192,0.614250


...

/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.244845,1.327277,0.549183,0.576046
2,0.231178,1.257643,0.579855,0.603257
3,0.236968,1.265970,0.587296,0.604444


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.38s/it]
/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.33it/s]
/Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.80it/s]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bi

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.13it/s]


,model_name,smoothing,eta,epochs,accuracy,macro_f1,worst_gap,macro_gap,model_dir
0,ls_gdro_eps01_eta01_3ep,0.10,0.10,3,0.574047,0.587065,0.311765,0.119478,models/challengers/ls_gdro_eps01_eta01_3ep
1,ls_gdro_eps005_eta005_3ep,0.05,0.05,3,0.588022,0.599825,0.335294,0.132326,models/challengers/ls_gdro_eps005_eta005_3ep
2,ls_gdro_eps01_eta005_3ep,0.10,0.05,3,0.585662,0.594196,0.335294,0.126925,models/challengers/ls_gdro_eps01_eta005_3ep


In [7]:
print(f"Summary saved to: {summary_path}")
summary_df


Summary saved to: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository/figures/challengers/c03_label_smoothing_groupdro_summary.csv


,model_name,smoothing,eta,epochs,accuracy,macro_f1,worst_gap,macro_gap,model_dir
0,ls_gdro_eps01_eta01_3ep,0.10,0.10,3,0.574047,0.587065,0.311765,0.119478,models/challengers/ls_gdro_eps01_eta01_3ep
1,ls_gdro_eps005_eta005_3ep,0.05,0.05,3,0.588022,0.599825,0.335294,0.132326,models/challengers/ls_gdro_eps005_eta005_3ep
2,ls_gdro_eps01_eta005_3ep,0.10,0.05,3,0.585662,0.594196,0.335294,0.126925,models/challengers/ls_gdro_eps01_eta005_3ep
